# 09. Relational Joins, Merges & Concatenations: Practical Fintech Guide

### 📌 Overview
Master **09. Relational Joins, Merges & Concatenations: Practical Fintech Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered:
- **Relational Merges (`pd.merge`)**: Inner, left, right, and full outer joins.
- **Multi-Table Joins**: 3-way and 4-way join pipelines across transactions, customers, merchants, and disputes.
- **Cardinality Validation & Anti-Joins**: Using `validate='m:1'` and `indicator=True` to find unmatched records.
- **Index-Based Joins (`df.join`)**: Fast joins on primary keys set as indices.
- **Concatenation (`pd.concat`)**: Vertical log batching (`axis=0`) and horizontal feature stacking (`axis=1`).
- **Fintech Real-World Scenarios**: Regulatory chargeback ratio monitoring, net merchant payout reconciliation, and KYC risk liability analysis.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Locate data directory
data_dir = 'data' if os.path.exists('data/raw_transactions.csv') else '../data'

# Load all 4 Fintech tables
df_tx = pd.read_csv(f'{data_dir}/raw_transactions.csv')
df_cust = pd.read_csv(f'{data_dir}/customers.csv')
df_merch = pd.read_csv(f'{data_dir}/merchants.csv')
df_disp = pd.read_csv(f'{data_dir}/disputes.csv')

print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv : {df_tx.shape[0]} rows, {df_tx.shape[1]} columns")
print(f"Loaded customers.csv        : {df_cust.shape[0]} rows, {df_cust.shape[1]} columns")
print(f"Loaded merchants.csv        : {df_merch.shape[0]} rows, {df_merch.shape[1]} columns")
print(f"Loaded disputes.csv         : {df_disp.shape[0]} rows, {df_disp.shape[1]} columns")


Pandas Version: 2.2.2
Loaded raw_transactions.csv : 15000 rows, 11 columns
Loaded customers.csv        : 4917 rows, 15 columns
Loaded merchants.csv        : 1897 rows, 11 columns
Loaded disputes.csv         : 10000 rows, 12 columns


### 🔹 Many-to-One Left Merges with `pd.merge()`
- **What it does:** Enrich individual transaction records with customer demographics, KYC verification status, and credit score.
- **Syntax:** `pd.merge(df_tx, df_cust, on='customer_id', how='left', validate='m:1')`
- **Operation:** `tx_with_cust = pd.merge(`
- **Key Note:** Use `validate='m:1'` when merging transactions (many) with customer profiles (one) to guarantee no duplicate keys exist in the customer dimension table.

In [2]:
# Enrich transactions with customer profile
tx_with_cust = pd.merge(
    df_tx,
    df_cust[['customer_id', 'first_name', 'last_name', 'kyc_status', 'risk_tier', 'credit_score', 'account_tier']],
    on='customer_id',
    how='left',
    validate='m:1'
)

print(f"Enriched shape: {tx_with_cust.shape}")
print("Sample Enriched Transactions:")
print(tx_with_cust[['transaction_id', 'customer_id', 'first_name', 'last_name', 'kyc_status', 'risk_tier', 'transaction_amount']].head())


Enriched shape: (15000, 17)
Sample Enriched Transactions:
  transaction_id customer_id first_name  last_name    kyc_status risk_tier  \
0       TX109326      C55082      Priya  Rodriguez       Pending    Medium   
1       TX106376      C76616       Paul   Anderson      Verified    Medium   
2       TX103301      C65296       John      Silva      Verified  Critical   
3       TX110701      C42098     Daniel   Thompson  Under Review       Low   
4       TX103284      C97782   Margaret   Robinson      Verified       Low   

   transaction_amount  
0              607.78  
1             1819.11  
2               64.08  
3             1025.73  
4              772.74  


### 🔹 Multi-Hop 3-Way Joins & Interchange Fee Calculation
- **What it does:** Chain multiple `pd.merge()` operations to connect transactions with both customer profiles and merchant business metadata, then compute the payment gateway's interchange fee revenue (`transaction_amount * interchange_fee_pct`).
- **Syntax:** `df_tx.merge(df_cust, on='customer_id').merge(df_merch, on='merchant_id')`
- **Operation:** `full_payments_df = (`

In [3]:
# 3-Way Join: Transactions + Customers + Merchants
full_payments_df = (
    df_tx.dropna(subset=['transaction_amount'])
    .merge(df_cust[['customer_id', 'kyc_status', 'risk_tier', 'account_tier']], on='customer_id', how='left')
    .merge(df_merch[['merchant_id', 'merchant_name', 'category', 'interchange_fee_pct', 'risk_rating']], on='merchant_id', how='left')
)

# Calculate payment processor interchange fee cut
full_payments_df['fee_revenue'] = full_payments_df['transaction_amount'] * full_payments_df['interchange_fee_pct']

# Group by merchant category to see transaction volume & revenue
category_summary = full_payments_df.groupby('category').agg(
    transaction_count=('transaction_id', 'count'),
    total_volume=('transaction_amount', 'sum'),
    fee_revenue=('fee_revenue', 'sum'),
    avg_fee_rate=('interchange_fee_pct', 'mean')
).sort_values(by='total_volume', ascending=False)

print("Merchant Category Revenue Summary:")
print(category_summary.round(2))


Merchant Category Revenue Summary:
                           transaction_count  total_volume  fee_revenue  \
category                                                                  
Food & Dining                           1846    1873309.73     38338.67   
Financial Services & SaaS               1696    1720558.94     45041.19   
Crypto & Digital Assets                 1734    1706025.91     66620.55   
Grocery & Supermarket                   1636    1681497.39     27061.76   
Travel & Airlines                       1618    1608480.48     47547.50   
Healthcare & Wellness                   1527    1573714.56     30626.45   
E-Commerce & Marketplaces               1530    1513838.76     38109.63   
Electronics & Computers                 1418    1412783.33     32511.96   
Gaming & Virtual Goods                  1246    1236726.40     40922.26   

                           avg_fee_rate  
category                                 
Food & Dining                      0.02  
Financial Ser

### 🔹 Chargeback & Dispute Merges (Joining Fact & Event Tables)
- **What it does:** Left join `raw_transactions` with `disputes` on `transaction_id` to correlate payment characteristics with dispute reasons, dispute statuses, and chargeback fees.
- **Syntax:** `pd.merge(df_tx, df_disp, on='transaction_id', how='left')`
- **Operation:** `tx_disputes = pd.merge(`

In [4]:
# Merge transactions with dispute records
tx_disputes = pd.merge(
    df_tx,
    df_disp[['transaction_id', 'dispute_id', 'dispute_date', 'dispute_reason', 'disputed_amount', 'dispute_status', 'chargeback_fee_usd']],
    on='transaction_id',
    how='left'
)

# Flag whether transaction was disputed
tx_disputes['is_disputed'] = tx_disputes['dispute_id'].notna()

print("Dispute Distribution by Card Type:")
dispute_by_card = tx_disputes.groupby('card_type').agg(
    total_transactions=('transaction_id', 'count'),
    disputed_transactions=('is_disputed', 'sum'),
    total_disputed_amount=('disputed_amount', 'sum')
)
dispute_by_card['dispute_rate_pct'] = (dispute_by_card['disputed_transactions'] / dispute_by_card['total_transactions']) * 100
print(dispute_by_card.round(2))


Dispute Distribution by Card Type:
            total_transactions  disputed_transactions  total_disputed_amount  \
card_type                                                                      
Amex                      4023                   1511             1383158.73   
Discover                  4053                   1585             1432818.99   
MasterCard                4065                   1623             1478672.67   
Visa                      4024                   1590             1426344.23   

            dispute_rate_pct  
card_type                     
Amex                   37.56  
Discover               39.11  
MasterCard             39.93  
Visa                   39.51  


### 🔹 Full Outer Join & Anti-Join with `indicator=True`
- **What it does:** Identify dormant / prospect customers who registered an account in `customers.csv` but have never made any transactions in `raw_transactions.csv`.
- **Syntax:** `pd.merge(df_tx, df_cust, on='customer_id', how='outer', indicator=True)`
- **Operation:** `outer_merged = pd.merge(`
- **Key Note:** Using `indicator=True` adds a `_merge` column (`'left_only'`, `'right_only'`, `'both'`) for filtering unmatched records.

In [5]:
# Outer merge with indicator
outer_merged = pd.merge(
    df_tx[['transaction_id', 'customer_id']],
    df_cust[['customer_id', 'first_name', 'last_name', 'account_created_at', 'account_tier', 'account_balance']],
    on='customer_id',
    how='outer',
    indicator=True
)

print("Merge Match Status Distribution:")
print(outer_merged['_merge'].value_counts())

# Filter for customers with ZERO transactions (Anti-Join)
dormant_customers = outer_merged[outer_merged['_merge'] == 'right_only'].drop_duplicates(subset=['customer_id'])
print(f"\nTotal Registered Customers with Zero Transactions: {len(dormant_customers)}")
print(dormant_customers[['customer_id', 'first_name', 'last_name', 'account_created_at', 'account_balance']].head())


Merge Match Status Distribution:
_merge
both          15000
right_only     3428
left_only         0
Name: count, dtype: int64

Total Registered Customers with Zero Transactions: 3428
   customer_id first_name last_name account_created_at  account_balance
22      C10088     Ashley   Sanchez         2021-09-19         15575.67
23      C10106      Linda     White         2022-10-17          2691.68
24      C10111     Carlos    Muller         2019-11-07         52580.64
25      C10141     Thomas    Garcia         2021-05-01         44666.89
31      C10204      Nancy   Sanchez         2024-02-26         29899.81


### 🔹 Index-Based Joins with `df.join()`
- **What it does:** When tables are indexed by their primary keys, `df.join()` provides a concise syntax for joining.
- **Syntax:** `df_tx.set_index('merchant_id').join(df_merch.set_index('merchant_id'))`
- **Operation:** `indexed_tx = df_tx.head(10).set_index('merchant_id')`

In [6]:
# Index-based join on merchant_id
indexed_tx = df_tx.head(10).set_index('merchant_id')
indexed_merch = df_merch.set_index('merchant_id')[['merchant_name', 'category', 'risk_rating']]

merchant_joined = indexed_tx.join(indexed_merch, how='left')
print("Index-Joined Transaction Records:")
print(merchant_joined[['transaction_id', 'transaction_amount', 'merchant_name', 'category', 'risk_rating']].head())


Index-Joined Transaction Records:
            transaction_id  transaction_amount     merchant_name  \
merchant_id                                                        
M3549             TX109326              607.78    Pioneer Direct   
M3068             TX106376             1819.11        Nova Cloud   
M3352             TX103301               64.08  Quantum Ventures   
M5807             TX110701             1025.73    Summit Express   
M7560             TX103284              772.74    Nexus Holdings   

                              category risk_rating  
merchant_id                                         
M3549          Electronics & Computers    Moderate  
M3068          Crypto & Digital Assets     Extreme  
M3352                    Food & Dining        High  
M5807                Travel & Airlines        High  
M7560        Financial Services & SaaS    Moderate  


### 🔹 Concatenation with `pd.concat()` (Vertical & Horizontal)
- **What it does:** - **Vertical Stacking (`axis=0`)**: Concatenates partitioned batches or incremental transaction logs.
- **Syntax:** `pd.concat([batch1, batch2], axis=0)` | `pd.concat([numeric_feats, categorical_feats], axis=1)`
- **Operation:** `batch_north = df_tx[df_tx['region'] == 'North'].head(50)`

In [7]:
# Vertical Concat: Combine 2 transaction partitions
batch_north = df_tx[df_tx['region'] == 'North'].head(50)
batch_south = df_tx[df_tx['region'] == 'South'].head(50)
combined_batches = pd.concat([batch_north, batch_south], axis=0, ignore_index=True)
print(f"Vertically Stacked Rows: {len(combined_batches)} (North: 50 + South: 50)")

# Horizontal Concat: Combine feature subsets side-by-side
num_features = df_tx[['transaction_amount', 'account_age_months']].head(5)
cat_features = df_tx[['card_type', 'device_type', 'transaction_status']].head(5)
feature_matrix = pd.concat([num_features, cat_features], axis=1)
print("\nHorizontally Combined Feature Matrix:")
print(feature_matrix)


Vertically Stacked Rows: 100 (North: 50 + South: 50)

Horizontally Combined Feature Matrix:
   transaction_amount  account_age_months card_type device_type  \
0              607.78                   8      Visa      Mobile   
1             1819.11                  28      Visa         POS   
2               64.08                  91      Visa      Mobile   
3             1025.73                  50      Amex     Desktop   
4              772.74                   5  Discover         POS   

  transaction_status  
0           Reversed  
1            Pending  
2          Completed  
3          Completed  
4             Failed  


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Scenario 1: Regulatory Chargeback Ratio Monitoring (Visa / Mastercard Thresholds)
- **Objective:** Scenario 1: Regulatory Chargeback Ratio Monitoring (Visa / Mastercard Thresholds)
- **Approach:** Apply built-in transformations and inspect output integrity.

In [8]:
# Calculate merchant-level dispute rate
merchant_tx_counts = df_tx.groupby('merchant_id').agg(
    total_tx=('transaction_id', 'count'),
    total_volume=('transaction_amount', 'sum')
).reset_index()

merchant_dsp_counts = df_disp.groupby('merchant_id').agg(
    dispute_count=('dispute_id', 'count'),
    total_disputed_amt=('disputed_amount', 'sum')
).reset_index()

# Merge metrics and join merchant metadata
merchant_compliance = (
    merchant_tx_counts
    .merge(merchant_dsp_counts, on='merchant_id', how='left')
    .fillna({'dispute_count': 0, 'total_disputed_amt': 0})
    .merge(df_merch[['merchant_id', 'merchant_name', 'category', 'risk_rating', 'is_chargeback_monitored']], on='merchant_id', how='left')
)

merchant_compliance['dispute_rate_pct'] = (merchant_compliance['dispute_count'] / merchant_compliance['total_tx']) * 100
merchant_compliance['exceeds_1pct_threshold'] = merchant_compliance['dispute_rate_pct'] > 1.0

high_risk_merchants = merchant_compliance[merchant_compliance['exceeds_1pct_threshold']].sort_values(by='dispute_rate_pct', ascending=False)
print(f"Total Merchants Monitored: {len(merchant_compliance)}")
print(f"Merchants Exceeding 1.0% Threshold: {len(high_risk_merchants)}")
print("\nTop High-Dispute Merchants Sample:")
print(high_risk_merchants[['merchant_id', 'merchant_name', 'category', 'total_tx', 'dispute_count', 'dispute_rate_pct', 'risk_rating']].head())


Total Merchants Monitored: 492
Merchants Exceeding 1.0% Threshold: 492

Top High-Dispute Merchants Sample:
    merchant_id      merchant_name                   category  total_tx  \
482       M9725          Nexus Pay  E-Commerce & Marketplaces        19   
402       M8273      Aero Holdings  E-Commerce & Marketplaces        19   
30        M1713  Horizon Solutions  Financial Services & SaaS        30   
243       M5743          Titan Hub      Grocery & Supermarket        33   
396       M8205      Vanguard Mart     Gaming & Virtual Goods        37   

     dispute_count  dispute_rate_pct risk_rating  
482             17         89.473684     Extreme  
402             17         89.473684    Moderate  
30              25         83.333333     Extreme  
243             27         81.818182         Low  
396             28         75.675676        High  


### 🔍 Scenario: Scenario 2: Net Merchant Settlement Reconciliation
- **Objective:** Scenario 2: Net Merchant Settlement Reconciliation
- **Approach:** Apply built-in transformations and inspect output integrity.

In [9]:
# Ingest & compute settlement ledger
tx_clean = df_tx.dropna(subset=['transaction_amount']).copy()
tx_enriched = tx_clean.merge(df_merch[['merchant_id', 'merchant_name', 'interchange_fee_pct']], on='merchant_id', how='left')
tx_enriched['interchange_fee'] = tx_enriched['transaction_amount'] * tx_enriched['interchange_fee_pct']

# Merchant gross & fees
settlement_ledger = tx_enriched.groupby(['merchant_id', 'merchant_name']).agg(
    gross_volume=('transaction_amount', 'sum'),
    total_interchange_fees=('interchange_fee', 'sum')
).reset_index()

# Chargeback deductions (Won by customer = merchant loss)
customer_won_disputes = df_disp[df_disp['dispute_status'] == 'Won - Customer'].groupby('merchant_id').agg(
    chargeback_loss=('disputed_amount', 'sum'),
    chargeback_fees=('chargeback_fee_usd', 'sum')
).reset_index()

# Merge settlement with deductions
settlement_summary = (
    settlement_ledger
    .merge(customer_won_disputes, on='merchant_id', how='left')
    .fillna({'chargeback_loss': 0, 'chargeback_fees': 0})
)

settlement_summary['net_merchant_payout'] = (
    settlement_summary['gross_volume']
    - settlement_summary['total_interchange_fees']
    - settlement_summary['chargeback_loss']
    - settlement_summary['chargeback_fees']
)

print("Merchant Settlement Ledger (Top 5 Payouts):")
print(settlement_summary.sort_values(by='net_merchant_payout', ascending=False).head().round(2))


Merchant Settlement Ledger (Top 5 Payouts):
    merchant_id    merchant_name  gross_volume  total_interchange_fees  \
212       M5128    Pulse Express      50504.15                 1186.85   
306       M6761  SilverLine Tech      48765.54                  955.80   
244       M5757       Global Hub      45356.37                  888.98   
319       M6952     Nexus Retail      44978.49                 1263.90   
175       M4178        Hyper Air      43337.19                  741.07   

     chargeback_loss  chargeback_fees  net_merchant_payout  
212             0.00              0.0             49317.30  
306          1208.71             35.0             46566.03  
244           362.54             35.0             44069.85  
319          3087.81             85.0             40541.78  
175          2317.73            100.0             40178.39  


### 🔍 Scenario: Scenario 3: Customer KYC Risk Tier vs Dispute Attribution
- **Objective:** Scenario 3: Customer KYC Risk Tier vs Dispute Attribution
- **Approach:** Apply built-in transformations and inspect output integrity.

In [10]:
# 4-Table Join Analysis
cross_risk_analysis = (
    df_tx.merge(df_cust[['customer_id', 'kyc_status', 'risk_tier', 'credit_score']], on='customer_id', how='left')
         .merge(df_merch[['merchant_id', 'category', 'risk_rating']], on='merchant_id', how='left')
         .merge(df_disp[['transaction_id', 'dispute_id', 'dispute_status', 'dispute_reason']], on='transaction_id', how='inner')
)

kyc_dispute_breakdown = cross_risk_analysis.groupby(['risk_tier', 'kyc_status']).agg(
    total_disputes=('dispute_id', 'count'),
    merchant_wins=('dispute_status', lambda s: (s == 'Won - Merchant').sum()),
    customer_wins=('dispute_status', lambda s: (s == 'Won - Customer').sum()),
    under_review=('dispute_status', lambda s: (s == 'Under Review').sum())
)
kyc_dispute_breakdown['merchant_win_rate_pct'] = (kyc_dispute_breakdown['merchant_wins'] / kyc_dispute_breakdown['total_disputes']) * 100

print("Customer Risk Tier & KYC Status vs Dispute Outcomes:")
print(kyc_dispute_breakdown.round(2))


Customer Risk Tier & KYC Status vs Dispute Outcomes:
                        total_disputes  merchant_wins  customer_wins  \
risk_tier kyc_status                                                   
Critical  Pending                  225             42             31   
          Rejected                 158             26             29   
          Under Review             143             27             24   
          Verified                 500             84             85   
High      Pending                  212             42             35   
          Rejected                 274             56             47   
          Under Review             178             32             29   
          Verified                 485             72             82   
Low       Pending                  339             49             47   
          Rejected                 391             57             59   
          Under Review             393             66             78   
          V